# Cereveil Fraud Detection Model Training — v2026

Train **ai4bharat/indic-bert** (ALBERT, ~33M params) on **30K** Indian digital payment SMS messages across 8 classes:
- `legitimate` (10K), `non_financial_spam`, `banking_phishing`, `upi_payment_fraud`
- `otp_credential_theft`, `kyc_identity_scam`, `investment_scam`, `loan_extortion`

**v2026 changes**: 10K legitimate samples (was 3K) with 15 subcategories covering real HDFC/SBI bank txns, telecom promos, govt advisories, app notifications, casual chat, encrypted banking. Includes 84 real false-positive messages from production. Goal: eliminate ~95% false positive rate on real-world bank SMS.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q -U transformers datasets accelerate scikit-learn seaborn sentencepiece

In [ ]:
# Cell 2 — GPU check + HuggingFace login
import torch
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("CEREVEIL_HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
del hf_token
print("Authenticated with Hugging Face using Kaggle Secrets")

In [ ]:
# Cell 3 — Load dataset
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict

# Set CEREVEIL_DATA_DIR when running outside Kaggle. On Kaggle, the
# notebook discovers an attached dataset containing the required JSONL files.
from pathlib import Path

required_files = {"train.jsonl", "validation.jsonl", "test.jsonl", "label_map.json"}
configured_data_dir = os.environ.get("CEREVEIL_DATA_DIR")

if configured_data_dir:
    DATA_DIR = Path(configured_data_dir)
else:
    candidates = [
        directory
        for directory in Path("/kaggle/input").rglob("*")
        if directory.is_dir() and required_files.issubset({item.name for item in directory.iterdir()})
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            "Attach exactly one Cereveil training dataset containing "
            "train.jsonl, validation.jsonl, test.jsonl, and label_map.json, "
            "or set CEREVEIL_DATA_DIR."
        )
    DATA_DIR = candidates[0]

print(f"Training data directory: {DATA_DIR}")

with open(f"{DATA_DIR}/label_map.json") as f:
    id2label = json.load(f)
id2label = {int(k): v for k, v in id2label.items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)
print(f"Labels ({num_labels}): {list(id2label.values())}")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
val_data = load_jsonl(f"{DATA_DIR}/validation.jsonl")
test_data = load_jsonl(f"{DATA_DIR}/test.jsonl")

print(f"Train: {len(train_data)}, Validation: {len(val_data)}, Test: {len(test_data)}")

dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data),
})
print(dataset)

# --- Visualize ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

all_records = train_data + val_data + test_data
class_counts = pd.Series([d["domain"] for d in all_records]).value_counts()
class_counts.plot(kind="barh", ax=axes[0], color=sns.color_palette("viridis", num_labels))
axes[0].set_title("Class Distribution (all splits)")
axes[0].set_xlabel("Count")

lang_counts = pd.Series([d["language"] for d in all_records]).value_counts()
lang_counts.plot(kind="bar", ax=axes[1], color=["#2196F3", "#FF9800", "#4CAF50"])
axes[1].set_title("Language Distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=0)

source_counts = pd.Series([d.get("source", "unknown") for d in all_records]).value_counts()
source_counts.plot(kind="bar", ax=axes[2], color=["#9C27B0", "#E91E63", "#00BCD4", "#FF5722"])
axes[2].set_title("Source Distribution")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Tokenize
from transformers import AutoTokenizer

MODEL_NAME = "ai4bharat/indic-bert"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
    )

tokenized = dataset.map(tokenize_fn, batched=True, batch_size=256)

cols_to_remove = [c for c in ["text", "domain", "language", "source", "confidence"] if c in tokenized["train"].column_names]
tokenized = tokenized.remove_columns(cols_to_remove)
tokenized.set_format("torch")

print(f"\nTokenized columns: {tokenized['train'].column_names}")
print(f"Sample shape: input_ids={tokenized['train'][0]['input_ids'].shape}")

In [ ]:
# Cell 5 — Load model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

print(f"Model: {MODEL_NAME}")
print(f"Architecture: {model.config.model_type}")
print(f"Total params: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable params: {trainable_params:,}")
print(f"Model size: {model_size_mb:.1f} MB")

In [ ]:
# Cell 6 — Training configuration
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

OUTPUT_DIR = "/kaggle/working/cereveil-fraud-checkpoints-v2026"

# 24K train / batch 32 = 750 steps/epoch
# 6 epochs * 750 = 4500 total steps (dataset is 2.5x larger so fewer epochs needed)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=6,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

steps_per_epoch = len(tokenized['train']) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * int(training_args.num_train_epochs)
print("Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")
print(f"  Steps/epoch: {steps_per_epoch}")
print(f"  Total steps: {total_steps}")

In [ ]:
# Cell 7 — Train
import time

print("Starting training...")
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final training loss: {train_result.training_loss:.4f}")

for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Cell 8 — Evaluate on test set
from sklearn.metrics import classification_report, confusion_matrix

test_output = trainer.predict(tokenized["test"])
test_preds = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
for key, value in test_output.metrics.items():
    print(f"  {key}: {value:.4f}")

target_names = [id2label[i] for i in range(num_labels)]
print("\n" + classification_report(test_labels, test_preds, target_names=target_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=target_names, yticklabels=target_names,
)
plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Live prediction test (focus on previous false positives)
import torch.nn.functional as F

SCAM_LABELS = {2, 3, 4, 5, 6, 7}

def predict(text, model=model, tokenizer=tokenizer):
    inputs = tokenizer(text, return_tensors="pt", max_length=MAX_LENGTH,
                       padding="max_length", truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits, dim=-1)
    pred_id = probs.argmax(dim=-1).item()
    confidence = probs[0, pred_id].item()
    label = id2label[pred_id]
    if pred_id in SCAM_LABELS:
        action = "ALERT"
    elif pred_id == 1:
        action = "IGNORE"
    else:
        action = "SAFE"
    return {"label": label, "confidence": confidence, "action": action}

test_messages = [
    # === REAL FALSE POSITIVES from the previous version (should all be SAFE now) ===
    "Available Bal in HDFC Bank A/c XX1292 as on yesterday:19-FEB-26 is INR 1,59,499.14. Cheques are subject to clearing.For updated A/C Bal dial 18002703333.",
    "Sent Rs.104.00\nFrom HDFC Bank A/C *1292\nTo SONU BHATT\nOn 17/02/26\nRef 118785042358\nNot You?\nCall 18002586161/SMS BLOCK UPI to 7308080808",
    "Credit Alert!\nRs.2.00 credited to HDFC Bank A/c XX1292 on 14-02-26 from VPA playstore1.bd@axisbank (UPI 114905850456)",
    "Update:\nRs. 20000.00 UPI mandate to Adobe Systems Software Ir has been cancelled from HDFC Bank A/c x1292.\nUMN: 133808b6e7ee48bb9bd9b8c5b3eb6201@okicici",
    "FD Renewed:\nHDFC Bank has renewed your FD.\nTo avoid TDS, submit Form 15G/H via NetBanking. Email advice will follow. (Not for NRIs).",
    "HDFCRMN EaSGfLd1mWk1xvufDzKL3AZJlDEcqKCSbt6Gtqa+X8dZdQWCc0cl2DI8NaNIEMCGU+NYnTO/RfkywMwgrkrVf+pKE3Ffs+2cZKpJmqoXjA==",
    "Mandate Set\nRs.12493.75\nFor Google Play\nFrom HDFC Bank A/c x1292\nUMN: aad483f6f9ac40ebb6b2b02461fd9414@ok\nNot you?\nCall 18002586161",
    # Telecom promos (were false positives)
    "IND vs NZ LIVE on JioHotstar! Plus enjoy 19+ OTTs with 12GB data for 30 days - all at just Rs. 195. Recharge your Airtel Prepaid now: https://i.airtel.in/rc195rchg",
    "Hi, we have processed Rs. 3599.0 for your Airtel Mobile 9319705039. The payment will be updated within 15 minutes.",
    "CONGRATULATIONS! You've unlocked flat Rs. 50 cashback on your airtel UPI transaction via Airtel Thanks app.",
    # Govt advisories (were false positives)
    "RBI Kehta Hai: Never share your OTP, PIN, CVV with anyone. Report fraud at cybercrime.gov.in or call 1930. -RBI",
    "Wondering how many SIMs have been issued in your name? Use Sanchar Saathi to know and report unauthorized connections.",
    # App notifs (were false positives)
    "ARRIVING: Blue Dart will deliver your shipment TODAY.Track https://acl.cc/BLUDRT/my3jbBAy .Be cautious of spam from unknown numbers.",
    "Refund of Rs 79 has been initiated for Swiggy order 215798679256204. Updated balance should reflect in 2 hours.",
    # Casual chat (were false positives)
    "ab problem ye hai ki mai WhatsApp pe likhoonga aur class hogyi toh gaali mujhe hi doge",
    "Dear Students, DAA Lab Test 1 will be conducted between 23rd to 28th February through the regular assessment mode.",
    # === ACTUAL SCAMS (should be ALERT) ===
    "Your SBI account has been blocked. Click here to verify: http://sbi-secure.xyz",
    "Aapka UPI se Rs.49,999 debit ho gaya hai. Cancel karne ke liye call karein: 9876543210",
    "Your OTP is 847293. Share this with our executive to complete verification.",
    "KYC expired! Update Aadhaar now or account will be frozen: bit.ly/kyc-update",
    "Earn Rs.50,000 daily from stock market. SEBI registered. Join now: wa.me/invest",
    "Aapne Rs 2,00,000 ka loan liya tha. Aaj raat tak pay karo warna police case hoga.",
    # === HARD NEGATIVES (legitimate, should be SAFE) ===
    "Your SBI account XX1234 has been credited with Rs.25,000.00 on 15-Jan. Bal: Rs.1,52,340.00",
    "Dear Customer, your OTP for net banking login is 493827. Valid for 5 minutes. Do not share with anyone.",
    "Reminder: EMI of Rs.12,450 for loan A/c XX7890 is due on 25-Jan. Pay via netbanking to avoid late fee.",
    "Bhai party ka paisa bhej de 500 rs UPI pe",
]

print(f"{'Message':<90} {'Prediction':<25} {'Conf':>6} {'Action':>6}")
print("=" * 135)

fp_count = 0
fn_count = 0
# First 16 should be SAFE, next 6 ALERT, last 4 SAFE
expected_actions = ["SAFE"]*16 + ["ALERT"]*6 + ["SAFE"]*4

for i, msg in enumerate(test_messages):
    result = predict(msg)
    display_msg = msg.replace('\n', ' ')[:87] + "..." if len(msg.replace('\n', ' ')) > 90 else msg.replace('\n', ' ')
    expected = expected_actions[i]
    ok = "" if result["action"] == expected else " <<<< WRONG"
    if result["action"] != expected:
        if expected == "SAFE":
            fp_count += 1
        else:
            fn_count += 1
    print(f"{display_msg:<90} {result['label']:<25} {result['confidence']:>5.1%} {result['action']:>6}{ok}")

print(f"\nFalse positives: {fp_count}/{16+4}  |  False negatives: {fn_count}/6")

In [ ]:
# Cell 10 — Save PyTorch model + tokenizer
import shutil

SAVE_DIR = "/kaggle/working/cereveil-fraud-model-v2026"
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.copy(f"{DATA_DIR}/label_map.json", f"{SAVE_DIR}/label_map.json")

# Save inference config
inference_config = {
    "model_name": "Cereveil fraud classifier v2026",
    "base_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "id2label": {str(k): v for k, v in id2label.items()},
    "label2id": label2id,
    "scam_label_ids": sorted(list(SCAM_LABELS)),
    "action_map": {
        "0": "SAFE", "1": "IGNORE",
        "2": "ALERT", "3": "ALERT", "4": "ALERT",
        "5": "ALERT", "6": "ALERT", "7": "ALERT",
    },
    "dataset_version": "v2026",
    "training_samples": len(train_data),
}
with open(f"{SAVE_DIR}/inference_config.json", "w") as f:
    json.dump(inference_config, f, indent=2)

print(f"Model saved to {SAVE_DIR}")
print("Contents:")
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{fname}")
    if size > 1e5:
        print(f"  {fname:<40} {size/1e6:.2f} MB")
    else:
        print(f"  {fname:<40} {size:,} bytes")

In [ ]:
# Cell 11 — Zip for download
import zipfile

zip_path = "/kaggle/working/cereveil-fraud-model-v2026.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(SAVE_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, SAVE_DIR)
            zf.write(file_path, arcname)

zip_size = os.path.getsize(zip_path) / 1e6
print(f"Zip: {zip_path} ({zip_size:.1f} MB)")
print("Download from Output tab.")